In [1]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine, euclidean
import polars as pl
import numpy as np

## Encoding

In [10]:
category_dict = {
    2: "Motoryzacja i pojazdy",
    1: "Film i animacja",
    10: "Muzyka",
    15: "Zwierzęta",
    17: "Sport",
    18: "Krótkie filmy",
    19: "Podróże i wydarzenia",
    20: "Gry",
    21: "Wideoblogi",
    22: "Ludzie i blogi",
    23: "Komedia",
    24: "Rozrywka",
    25: "Wiadomości i polityka",
    26: "Poradniki i styl życia",
    27: "Edukacja",
    28: "Nauka i technologie",
    29: "Organizacje non-profit i aktywizm",
    30: "Filmy",
    31: "Anime / Animacja",
    32: "Akcja / Przygodowe",
    33: "Klasyki",
    34: "Komedia",
    35: "Dokument",
    36: "Dramat",
    37: "Familijne",
    38: "Zagraniczne",
    39: "Horror",
    40: "Sci-Fi / Fantasy",
    41: "Thriller",
    42: "Krótkometrażowe",
    43: "Programy",
    44: "Zwiastuny"
}

emotion_dict = {
  "neutral" : "Neutralność",
  "joy" : "Radość", 
  "anger" : "Złość",
  "sadness" : "Smutek",
  "sarcasm" : "Sarkazm",
  "curiosity" : "Ciekawość",
  "gratitude": "Wdzięczność"
}

data = pl.read_csv("../src/src_files/data/comments_emotions.csv").with_columns(
    pl.col("CategoryID")
        .map_elements(category_dict.get, return_dtype=pl.String)
        .cast(pl.Categorical),
    pl.col("Emotions")
        .replace_strict(emotion_dict, return_dtype=pl.String)
        .cast(pl.Categorical)
    )

In [17]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

In [27]:
test_data = data["CommentText"].to_list()

In [28]:
result = model.encode(sentences=test_data, batch_size=32)

In [30]:
np.savetxt("../src/src_files/data/embeddings.txt", result)

## Centroids

In [3]:
results = np.loadtxt("../src/src_files/data/embeddings.txt", np.float32)

In [4]:
def select_centroid(
    df: pl.DataFrame,
    column : str,
    key: str,
    results: np.ndarray
    ) -> np.ndarray:
    
    indices = (
        df
        .with_row_index()
        .filter(pl.col(column) == key)
        .select("index")
        .to_series()
        .to_list()
    )

    emotions_results = results[indices]

    return  np.mean(emotions_results, axis=0)


In [5]:
centroids = {emotions:select_centroid(data, "Emotions", emotions, results) for emotions in data["Emotions"].unique()}

In [6]:
category_centroids = {cat:select_centroid(data, "CategoryID",cat, results) for cat in data["CategoryID"].unique()}

In [7]:
def create_distance_matrix(
        centroids: dict[str, np.float32],
        name: str
) -> pl.DataFrame:
    distances : list[tuple[str, str, np.float64]]= []

    for x in centroids:
        for y in centroids:
            distances.append(
                (x,y,cosine(centroids[x], centroids[y]))
            )

    distance_matrix = (
        pl.DataFrame(distances,
                    schema=["x", name, "distance"],
                    orient="row")
        .pivot(on="x", index=name, values="distance")
    )
    return distance_matrix


In [8]:
distance_matrix = create_distance_matrix(centroids, "Emocje")

In [9]:
cats_dm = create_distance_matrix( category_centroids, "Kategorie")

In [47]:
cats_dm.write_csv("../src/src_files/data/categories_distance_matrix.csv")

In [10]:
distance_matrix.write_csv("../src/src_files/data/emotions_distance_matrix.csv")

## Most common words & PMI

In [14]:
from wordcloud import STOPWORDS
import spacy
import polars as pl

In [15]:
data = pl.read_csv("../src/src_files/data/comments_emotions.csv")#.sample(100_000)

In [16]:
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])
LEN_DATASET = len(data)
def generate_most_common_words(
    data: pl.DataFrame,
    column: str
) -> pl.DataFrame:
    common_words = (
        data
        .lazy()
        .select(
            pl.col(column),
            pl.col("CommentText")
                .str.to_lowercase()
                .str.extract_all(r"[^\W\d_]+")
                .list.unique()
                .alias("Word")
        )
        .explode("Word")
        .group_by(
            pl.col(column),
            pl.col("Word")
        )
        .len()
        .filter(
            pl.col("Word").is_in(STOPWORDS).not_(),
            pl.col("Word").str.len_chars() > 2,
            pl.col("len") > 50
        )
    ).collect()

    lemmatized = []
    for token in nlp.pipe(common_words["Word"]):
        lemmatized.append(token[0].lemma_)

    common_words = (common_words
        .with_columns(
            Word = pl.Series(lemmatized)
        )
        .group_by(
            pl.col(column),
            pl.col("Word")
        )
        .sum()
    )

    return common_words

In [17]:
common_words_emotions   = generate_most_common_words(data, "Emotions")
common_words_categories = generate_most_common_words(data, "CategoryID")

In [18]:
common_words_emotions.write_csv("../src/src_files/data/common_words_emotions.csv")
common_words_categories.write_csv("../src/src_files/data/common_words_categories.csv")

In [38]:
em_count = data["Emotions"].value_counts(name="EmCount",normalize=True)
w_count = common_words_emotions.group_by("Word").agg(pl.col("len").sum().alias("SumWord"))

pmi = (
    common_words_emotions
    .join(
        em_count,
        on="Emotions"
    )
    .join(
        w_count,
        on="Word"
    )
    .with_columns(
        #PMI = ((pl.col("len"))/(pl.col("EmCount")*pl.col("SumWord"))).log(base = 2) * (pl.col("len") / pl.col("SumWord")).log()
        PMI = ((pl.col("len"))/(pl.col("EmCount")*pl.col("SumWord"))).log(base = 2)  / (len(data)/pl.col("len")).log(base=2)
    )
    .select(
        pl.col("Emotions"),
        pl.col("Word"),
        pl.col("PMI")
    )
)

In [41]:
pmi.sort(by="PMI", descending=True).head(10)

Emotions,Word,PMI
str,str,f64
"""gratitude""","""thank""",0.753229
"""sadness""","""sad""",0.526543
"""anger""","""angry""",0.399463
"""anger""","""piss""",0.392982
"""neutral""","""neutral""",0.392361
"""sadness""","""cry""",0.382885
"""sadness""","""sadly""",0.38186
"""sadness""","""heartbreake""",0.380225
"""anger""","""mad""",0.365809


In [42]:
(pmi
    .with_columns(
        Emotions = pl.col("Emotions")
            .replace_strict(emotion_dict,
                return_dtype=pl.String)
    )
    .write_csv("../src/src_files/data/pmi.csv"))